In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.master('local[*]').appName('14').getOrCreate()

In [3]:
df = spark.read.json('C:\\datas\\3rd_week_data\\faces_detection.json')

In [4]:
df.show(5)

+--------------------+--------------------+------+
|          annotation|             content|extras|
+--------------------+--------------------+------+
|[{333, 650, [Face...|http://com.datatu...|  NULL|
|[{697, 1280, [Fac...|http://com.datatu...|  NULL|
|[{240, 460, [Face...|http://com.datatu...|  NULL|
|[{291, 600, [Face...|http://com.datatu...|  NULL|
|[{326, 580, [Face...|http://com.datatu...|  NULL|
+--------------------+--------------------+------+
only showing top 5 rows



In [5]:
df.printSchema()

root
 |-- annotation: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- imageHeight: long (nullable = true)
 |    |    |-- imageWidth: long (nullable = true)
 |    |    |-- label: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- notes: string (nullable = true)
 |    |    |-- points: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- x: double (nullable = true)
 |    |    |    |    |-- y: double (nullable = true)
 |-- content: string (nullable = true)
 |-- extras: string (nullable = true)



image Classification(이미지 입력시 얼굴이냐 아니냐 분류만 함)과 image Detection의 차이 -> image Detection은 Localization(얼굴의 위치도 찾음) + Classification(찾은 위치가 뭔지도 알려줌)

- **content**: 얼굴이 포함된 원본 이미지 (`StringType`)
- **annotation**: 원본 이미지에서 얼굴들의 정보, 하나의 이미지로부터 여러 개의 얼굴이 검출될 수 있으므로, `ArrayType` (**annotation** array를 구성하는 각 요소의 type은 `StructType`)
> - **imageHeight**, **imageWidth**: 원본 이미지의 높이와 폭
> - **label**: 얼굴 또는 X
> - **points**: 원본 이미지 내에서의 검출 영역 좌표 `(x,y)`
>> - 영역을 표현하기 위해서는 두 개의 좌표가 필요하므로 `ArrayType`
>> - `(x,y)` 좌표는 각 요소가 `DoubleType`인 `StructType`
- **DataFrame의 하나의 행**에는 **하나의 원본이미지(content)로부터 추출한 얼굴 데이터들이** 포함되어 있으며, 행마다 실제 포함하고 있는 얼굴 데이터의 수는 다를 수 있음

In [8]:
# annotation 확인
# 첫번째 이미지는 두 개의 얼굴 포함, 두번째 이미지는 한 개의 얼굴 포함
df.select('annotation').show(2,truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|annotation                                                                                                                                                                                                                |
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[{333, 650, [Face], , [{0.08615384615384615, 0.3063063063063063}, {0.1723076923076923, 0.45345345345345345}]}, {333, 650, [Face], , [{0.583076923076923, 0.2912912912912913}, {0.6584615384615384, 0.46846846846846846}]}]|
|[{697, 1280, [Face], , [{0.7053087757313109, 0.23260437375745527}, {0.7692307692307693, 0.36182902584493043}]}]    

In [9]:
# 얼굴의 points 확인
df.select(df.annotation.points).show(2,truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|annotation.points                                                                                                                                                             |
+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|[[{0.08615384615384615, 0.3063063063063063}, {0.1723076923076923, 0.45345345345345345}], [{0.583076923076923, 0.2912912912912913}, {0.6584615384615384, 0.46846846846846846}]]|
|[[{0.7053087757313109, 0.23260437375745527}, {0.7692307692307693, 0.36182902584493043}]]                                                                                      |
+------------------------------------------------------------------------------------------------------------------

In [11]:
# 각 원본이미지(행)에 대해 두번째 객체의 좌표 정보
# 두번째 객체란 annotation[1]
df.select(df.annotation[1].points).show(2,truncate=False)
# 두번째 원본이미지(2행)은 하나의 얼굴 객체만 있으므로 NULL

+------------------------------------------------------------------------------------+
|annotation[1].points                                                                |
+------------------------------------------------------------------------------------+
|[{0.583076923076923, 0.2912912912912913}, {0.6584615384615384, 0.46846846846846846}]|
|NULL                                                                                |
+------------------------------------------------------------------------------------+
only showing top 2 rows



In [14]:
# 각 원본 이미지의 첫번째 객체 y좌표만 추출
df.select(df.annotation[0].points['y']).show(2,truncate=False)

+------------------------------------------+
|annotation[0].points.y                    |
+------------------------------------------+
|[0.3063063063063063, 0.45345345345345345] |
|[0.23260437375745527, 0.36182902584493043]|
+------------------------------------------+
only showing top 2 rows



#### 위 데이터 값으로부터 각 annotation에 대해 **실제 좌표값**을 계산해보자